# Masking por bloque con cinco semillas, DS2OS

Compara el masking por bloque con el control. Se repite con las semillas 42, 1, 2, 3 y 4. La partición de los datos es fija, de modo que solo varía el azar del entrenamiento.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

In [2]:
DATA_PATH = Path.home() / ".cache/kagglehub/datasets/libamariyam/ds2os-dataset/versions/1/DS2OS.csv"

df = pd.read_csv(DATA_PATH).drop(columns=['timestamp'])
df['accessedNodeType'] = df['accessedNodeType'].fillna('missing')
df['value'] = df['value'].replace({'true': 1, 'false': 0, 'twenty': 20, 'none': 0})
df['value'] = pd.to_numeric(df['value'], errors='coerce').fillna(0)
print(df.shape)

(357952, 12)


In [3]:
CAT_COLS = ['sourceID', 'sourceAddress', 'sourceType', 'sourceLocation',
            'destinationServiceAddress', 'destinationServiceType', 'destinationLocation',
            'accessedNodeAddress', 'accessedNodeType', 'operation']

y = (df['normality'] != 'normal').astype(int).values
X = pd.get_dummies(df.drop(columns=['normality']), columns=CAT_COLS)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

scaler = MinMaxScaler().fit(X_train)
X_train_arr = scaler.transform(X_train)
X_test_arr = scaler.transform(X_test)
X_train_normal = X_train_arr[y_train == 0]

print(f"X_train: {X_train_arr.shape}, X_train_normal: {X_train_normal.shape}, X_test: {X_test_arr.shape}")

X_train: (286361, 505), X_train_normal: (278347, 505), X_test: (71591, 505)


In [4]:
# Cada columna one-hot pertenece al bloque de su variable original y 'value' forma su propio bloque
columns = list(X.columns)
group_of_column = []
for col in columns:
    matched = None
    for cat in CAT_COLS:
        if col.startswith(cat + '_'):
            matched = cat
            break
    group_of_column.append(matched if matched is not None else col)

groups = sorted(set(group_of_column), key=lambda g: group_of_column.index(g))
group_index = {g: i for i, g in enumerate(groups)}
num_groups = len(groups)

group_matrix = np.zeros((num_groups, len(columns)), dtype=np.float32)
for col_idx, g in enumerate(group_of_column):
    group_matrix[group_index[g], col_idx] = 1.0

print(f"Columnas: {len(columns)}, bloques: {num_groups}")
print(f"Bloques: {groups}")
print(f"Tamaño de cada bloque: {group_matrix.sum(axis=1).astype(int).tolist()}")

Columnas: 505, bloques: 11
Bloques: ['value', 'sourceID', 'sourceAddress', 'sourceType', 'sourceLocation', 'destinationServiceAddress', 'destinationServiceType', 'destinationLocation', 'accessedNodeAddress', 'accessedNodeType', 'operation']
Tamaño de cada bloque: [1, 84, 89, 8, 21, 85, 8, 21, 170, 13, 5]


In [5]:
class BlockMasking(layers.Layer):
    """Anula con probabilidad mask_rate bloques completos de columnas.

    group_matrix (num_groups, input_dim) indica a qué bloque pertenece cada columna.
    """
    def __init__(self, mask_rate, group_matrix, **kwargs):
        super().__init__(**kwargs)
        self.mask_rate = mask_rate
        self.group_matrix_np = group_matrix
        self.group_matrix = tf.constant(group_matrix, dtype=tf.float32)
        self.num_groups = group_matrix.shape[0]

    def call(self, inputs, training=None):
        if training:
            batch_size = tf.shape(inputs)[0]
            group_keep = tf.cast(
                tf.random.uniform((batch_size, self.num_groups)) >= self.mask_rate,
                tf.float32,
            )
            col_keep = tf.matmul(group_keep, self.group_matrix)
            return inputs * col_keep
        return inputs

    def get_config(self):
        config = super().get_config()
        config.update({'mask_rate': self.mask_rate})
        return config


def build_dae_block_masking(input_dim, hidden_units, mask_rate, group_matrix):
    inputs = keras.Input(shape=(input_dim,))
    x = BlockMasking(mask_rate, group_matrix)(inputs)
    for units in hidden_units[:-1]:
        x = layers.Dense(units, activation='relu')(x)
    x = layers.Dense(hidden_units[-1], activation='relu')(x)
    for units in reversed(hidden_units[:-1]):
        x = layers.Dense(units, activation='relu')(x)
    outputs = layers.Dense(input_dim, activation='sigmoid')(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='mse')
    return model


def reconstruction_error(model, X):
    return np.mean(np.square(X - model.predict(X, verbose=0)), axis=1)


HIDDEN_UNITS = [128, 64, 16]
MASK_RATES = [0.0, 0.1, 0.2, 0.3]  # 0.0 = control sin corrupción
SEEDS = [42, 1, 2, 3, 4]  # la partición de los datos se mantiene fija con SEED = 42

results = {}
for seed in SEEDS:
    for rate in MASK_RATES:
        tf.keras.utils.set_random_seed(seed)
        model = build_dae_block_masking(X_train_arr.shape[1], HIDDEN_UNITS, rate, group_matrix)
        history = model.fit(
            X_train_normal, X_train_normal,
            epochs=100,
            batch_size=256,
            validation_split=0.1,
            callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)],
            verbose=0,
        )
        auc = roc_auc_score(y_test, reconstruction_error(model, X_test_arr))
        results[(seed, rate)] = auc
        print(f"seed={seed} mask_rate={rate}: epocas={len(history.history['loss'])}, ROC-AUC={auc:.4f}", flush=True)

seed=42 mask_rate=0.0: epocas=99, ROC-AUC=0.9211


seed=42 mask_rate=0.1: epocas=100, ROC-AUC=0.8933


seed=42 mask_rate=0.2: epocas=100, ROC-AUC=0.9548


seed=42 mask_rate=0.3: epocas=100, ROC-AUC=0.8033


seed=1 mask_rate=0.0: epocas=28, ROC-AUC=0.8431


...

seed=3 mask_rate=0.3: epocas=100, ROC-AUC=0.8414


seed=4 mask_rate=0.0: epocas=68, ROC-AUC=0.9370


seed=4 mask_rate=0.1: epocas=100, ROC-AUC=0.8187


seed=4 mask_rate=0.2: epocas=100, ROC-AUC=0.9261


seed=4 mask_rate=0.3: epocas=100, ROC-AUC=0.8492


In [6]:
import pandas as pd

df_res = pd.Series(results).unstack()  # filas: semilla, columnas: mask_rate
df_res.index.name = 'seed'
print("AUC por semilla y nivel de corrupcion:")
print(df_res.round(4).to_string())

print("\nMedia y desviacion tipica (5 semillas):")
for rate in MASK_RATES:
    v = df_res[rate]
    print(f"  mask_rate={rate}: media={v.mean():.4f}  std={v.std(ddof=1):.4f}  min={v.min():.4f}  max={v.max():.4f}")

print("\nDiferencia con el control de la misma semilla:")
for rate in MASK_RATES[1:]:
    d = df_res[rate] - df_res[0.0]
    print(f"  mask_rate={rate}: media={d.mean():+.4f}  std={d.std(ddof=1):.4f}  "
          f"semillas que mejoran al control={int((d > 0).sum())}/{len(d)}")

AUC por semilla y nivel de corrupcion:
         0.0     0.1     0.2     0.3
seed                                
1     0.8431  0.9238  0.8429  0.8733
2     0.9168  0.9505  0.8607  0.8549
3     0.8642  0.9375  0.9577  0.8414
4     0.9370  0.8187  0.9261  0.8492
42    0.9211  0.8933  0.9548  0.8033

Media y desviacion tipica (5 semillas):
  mask_rate=0.0: media=0.8964  std=0.0405  min=0.8431  max=0.9370
  mask_rate=0.1: media=0.9048  std=0.0526  min=0.8187  max=0.9505
  mask_rate=0.2: media=0.9084  std=0.0535  min=0.8429  max=0.9577
  mask_rate=0.3: media=0.8444  std=0.0258  min=0.8033  max=0.8733

Diferencia con el control de la misma semilla:
  mask_rate=0.1: media=+0.0083  std=0.0828  semillas que mejoran al control=3/5
  mask_rate=0.2: media=+0.0120  std=0.0557  semillas que mejoran al control=2/5
  mask_rate=0.3: media=-0.0520  std=0.0577  semillas que mejoran al control=1/5
